# Predicción de Quiebra Empresarial con Aprendizaje de Máquina
### Dataset: Taiwanese Bankruptcy Prediction (UCI ID: 572)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/REPO/blob/main/Proyecto_Final_Quiebra_Taiwan.ipynb)

- **Integrantes:**
  - Wanerge Almanza Velasquez
  - Jhon Fredy Hoyos Cardenas
  - Miguel Angel Castiblanco Florez
- **Materia:** Modelos y Simulación de Sistemas II
- **Institución:** Universidad de Antioquia (UdeA) - Medellín, Colombia
- **Docente:** Julián D. Arias-Londoño

---
### Estructura del Trabajo
El presente notebook implementa el ciclo completo de desarrollo en Machine Learning acorde a la **Guía del Proyecto Final** y a la teoría vista durante el curso:
1. **Configuración inicial y Registro de Decisiones Metodológicas**
2. **Descripción del Problema y Análisis Exploratorio de Datos (EDA) [10%]**
3. **Estado del Arte y Justificación de Métricas [10%]**
4. **Entrenamiento y Evaluación de Modelos (5 familias obligatorias) [15%]**
5. **Reducción de Dimensionalidad (Análisis univariado, PCA y UMAP) [15%]**
6. **Modelos Complementarios y Explicabilidad (XAI con SHAP)**
7. **Discusión General y Conclusiones Comparativas**


## 1. Configuración del Entorno y Carga de Datos

> **¿Qué hacemos y por qué?**
> Instalamos las dependencias requeridas en caso de ejecutar en Google Colab (`ucimlrepo`, `imbalanced-learn`, `umap-learn`, `shap`), fijamos la semilla de reproducibilidad (`SEED = 42`) y definimos una función de carga dual: si el archivo `data/data.csv` existe localmente se carga directo; de lo contrario, se descarga de forma automatizada desde el repositorio oficial UCI.


In [1]:
# Instalación de librerías para entorno Google Colab (se ejecuta silenciosamente)
!pip install -q ucimlrepo imbalanced-learn umap-learn shap

import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración gráfica global
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = False

# Semilla global de reproducibilidad (enseñada a lo largo de los laboratorios del curso)
SEED = 42
np.random.seed(SEED)

print("Entorno inicializado correctamente con SEED =", SEED)


In [2]:
# Función de carga dual: local o repositorio oficial UCI
def cargar_dataset():
    ruta_local = os.path.join("data", "data.csv")
    if os.path.exists(ruta_local):
        print(f"[INFO] Cargando dataset desde almacenamiento local: {ruta_local}")
        df = pd.read_csv(ruta_local)
    else:
        print("[INFO] Archivo local no encontrado. Descargando automáticamente de UCI Machine Learning Repository (ID 572)...")
        from ucimlrepo import fetch_ucirepo
        taiwan_data = fetch_ucirepo(id=572)
        X = taiwan_data.data.features
        y = taiwan_data.data.targets
        df = pd.concat([y, X], axis=1)
        print("[INFO] Descarga completada con éxito desde UCI.")
        
    # Limpieza de espacios en blanco en los nombres de las columnas
    df.columns = [c.strip() for c in df.columns]
    
    # Normalización de caracteres especiales (por ejemplo el símbolo de moneda Yuan ¥)
    rename_cols = {}
    for col in df.columns:
        if "Yuan" in col:
            rename_cols[col] = col.replace("", "¥").replace("?", "")
    df.rename(columns=rename_cols, inplace=True)
    return df

df_raw = cargar_dataset()
print(f"Dimensiones del conjunto de datos: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")


## 2. Descripción del Problema y Análisis Exploratorio de Datos (10%)

### 2.1 Contexto del Problema y Justificación de Machine Learning
La insolvencia financiera o bancarrota empresarial es un fenómeno crítico con repercusiones macroeconómicas severas en el crédito, el empleo y los mercados bursátiles. La detección anticipada de señales de estrés financiero permite a instituciones financieras, inversores y reguladores mitigar riesgos de impago.

Históricamente, el análisis de bancarrota se abordó con modelos paramétricos lineales clásicos como el **Z-Score de Altman (1968)** (basado en Análisis Discriminante Lineal - LDA) o el **O-Score de Ohlson (1980)** (basado en Regresión Logística). Sin embargo, dichos enfoques adolecen de severas limitaciones en escenarios corporativos reales:
1. **Supuestos distributivos restrictivos:** Exigen normalidad multivariada y homocedasticidad, premisas violadas por los ratios contables que presentan colas pesadas y asimetrías extremas.
2. **Incapacidad ante relaciones no lineales:** La quiebra emerge de interacciones complejas entre endeudamiento, rotación de activos y liquidez a corto plazo.
3. **Asimetría extrema de costos de error:** Clasificar erróneamente a una empresa en quiebra como solvente (Falso Negativo) ocasiona pérdidas crediticias devastadoras frente al costo de investigar una empresa solvente sospechosa (Falso Positivo).

Por tanto, el empleo de **Machine Learning supervisado** se justifica plenamente al permitir modelar fronteras de decisión no lineales complejas sin requerir supuestos distributivos restrictivos, integrando esquemas de regularización y remuestreo para gestionar el desbalance y la asimetría de costos.


In [3]:
# Verificación de integridad: datos faltantes, tipos de datos y duplicados
print("--- AUDITORÍA DE CALIDAD DE DATOS ---")
print(f"Valores nulos en todo el dataset: {df_raw.isna().sum().sum()}")
print(f"Filas duplicadas: {df_raw.duplicated().sum()}")
print(f"Tipos de datos detectados:
{df_raw.dtypes.value_counts()}")


### 2.2 Auditoría de Características y Detección de Columnas Constantes

> **Decisión Metodológica (Filtro preliminar):**
> Al inspeccionar la cantidad de valores únicos por columna, se detecta que la columna `Net Income Flag` posee exactamente **un único valor** en las 6,819 observaciones (todos los valores son 1). Una variable con varianza cero no aporta ninguna información discriminativa a ningún algoritmo y debe ser eliminada del espacio de predictores. Por otro lado, la variable binaria `Liability-Assets Flag` posee solo 8 ocurrencias con valor 1 (de las cuales 6 derivaron en quiebra), actuando como una señal poco frecuente pero de alta sospecha de insolvencia.


In [4]:
# Análisis de valores únicos y eliminación de columna constante
unique_counts = df_raw.nunique()
const_cols = unique_counts[unique_counts <= 1].index.tolist()
print(f"Columnas con varianza cero (constantes): {const_cols}")

# Eliminar columna constante
df = df_raw.drop(columns=const_cols).copy()
print(f"Columnas restantes para entrenamiento: {df.shape[1] - 1} predictores + 1 variable objetivo ('Bankrupt?')")


### 2.3 Análisis de Desbalance Severo de Clases

> **¿Qué observamos?**
> La variable objetivo `Bankrupt?` presenta 6,599 observaciones de empresas solventes (96.77%) y solo 220 registros de empresas en quiebra (3.23%). Esto representa una relación de desbalance cercana a **30 a 1**.
> 
> **Consecuencia metodológica crucial:** La métrica convencional de **Exactitud (Accuracy)** resulta totalmente engañosa e inadmisible como criterio de selección; un clasificador trivial que prediga siempre clase 0 obtendría un 96.77% de accuracy, pero tendría un 0% de sensibilidad en la detección de empresas en quiebra.


In [5]:
# Visualización del desbalance de clases
clase_0 = (df['Bankrupt?'] == 0).sum()
clase_1 = (df['Bankrupt?'] == 1).sum()
n_total = len(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ['#2b5c8f', '#d9534f']

# 1. Gráfico de barras
bars = axes[0].bar(['Solvente (0)', 'Quiebra (1)'], [clase_0, clase_1], color=colors, width=0.5, edgecolor='black')
axes[0].set_title('Figura 1a: Frecuencia Absoluta de Empresas por Clase', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de Observaciones', fontsize=10)
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 120, f'{int(yval):,}
({yval/n_total*100:.2f}%)', 
                 ha='center', va='bottom', fontsize=9.5, fontweight='bold')
axes[0].set_ylim(0, 7500)

# 2. Gráfico de dona
axes[1].pie([clase_0, clase_1], labels=['Solvente (96.77%)', 'Quiebra (3.23%)'], 
            autopct='%1.2f%%', startangle=30, colors=colors, explode=(0, 0.15),
            textprops={'fontsize': 10, 'fontweight': 'bold'}, wedgeprops=dict(width=0.6, edgecolor='black'))
axes[1].set_title('Figura 1b: Proporción Relativa de Clases (Desbalance 30:1)', fontsize=11, fontweight='bold')

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig(os.path.join("figures", "fig1_desbalance_clases.png"), dpi=300, bbox_inches='tight')
plt.show()


### 2.4 Análisis de Escala Extrema, Asimetría y Outliers Contables

> **¿Qué observamos?**
> A diferencia de lo reportado a menudo en la literatura informal, este dataset **no tiene todas las variables normalizadas entre [0, 1]**. 
> Se comprueba que **24 columnas** presentan valores que superan la unidad, alcanzando magnitudes del orden de $10^9$ y $10^{10}$ (como `Cash Turnover Rate`, `Quick Assets/Current Liability`, `Total debt/Total net worth` y tasas de rotación de inventario). Asimismo, **62 de las 94 variables predictoras** presentan un coeficiente de asimetría absoluta ($|skew| > 5$).
> 
> **Decisión Metodológica:**
> Un escalado convencional mediante `StandardScaler` (que depende de la media y la desviación estándar) se distorsionará fuertemente por estos valores extremos. Por ello, para los modelos sensibles a la escala (KNN, Regresión Logística, MLP, SVM) se adoptará un **`QuantileTransformer` con salida normal**, el cual mapea las variables a una distribución gaussiana estándar robusta ante outliers extremos sin perder orden de rango.


In [6]:
# Detección de variables con rangos extremos fuera de [0, 1]
cols_pred = [c for c in df.columns if c != "Bankrupt?"]
mm = df[cols_pred].agg(["min", "max"]).T
out_unit = mm[(mm["min"] < 0) | (mm["max"] > 1)]
print(f"Total de características con valores fuera de [0, 1]: {len(out_unit)}")
print("
Top 8 características con mayores valores máximos observados:")
print(out_unit.sort_values(by="max", ascending=False).head(8))

# Inspección visual de outliers mediante diagramas de caja (Boxplots)
sample_outlier_cols = [
    'Cash Turnover Rate',
    'Quick Assets/Current Liability',
    'Total debt/Total net worth',
    'Net Value Growth Rate'
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for idx, col in enumerate(sample_outlier_cols):
    ax = axes[idx // 2, idx % 2]
    sns.boxplot(data=df, x='Bankrupt?', y=col, ax=ax, palette=['#2b5c8f', '#d9534f'], showmeans=True,
                meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
    ax.set_title(f'{col}
(Máx: {df[col].max():.2e} | Skew: {df[col].skew():.2f})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Estado (0: Solvente, 1: Quiebra)', fontsize=9)
    ax.set_ylabel('Valor de Ratio Contable', fontsize=9)

plt.suptitle('Figura 2: Distribución de Ratios Financieros con Escala Extrema y Outliers', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join("figures", "fig2_asimetria_outliers.png"), dpi=300, bbox_inches='tight')
plt.show()


### 2.5 Análisis de Redundancia y Multicolinealidad

> **¿Qué observamos?**
> En contabilidad y finanzas, numerosos ratios miden variantes muy similares de la rentabilidad sobre activos (ROA), liquidez corriente y valor en libros. 
> El análisis de correlación de Pearson sobre las 94 variables predictoras revela **28 pares de características con correlación extrema $|r| \ge 0.90$**, incluyendo 12 pares con $|r| \ge 0.99$ (por ejemplo, variantes de `Operating Gross Margin`, `Net Value Per Share` y `Current Liabilities/Equity`).
> 
> **Decisión Metodológica:**
> Esta alta multicolinealidad motiva formalmente el estudio de **Reducción de Dimensionalidad** en la Sección 5 (mediante filtros univariados, PCA lineal y variedades no lineales con UMAP) para verificar si es posible condensar el espacio sin degradar la capacidad predictiva.


In [7]:
# Cálculo de correlaciones entre variables predictoras
corr_pred = df[cols_pred].corr().abs()
iu = np.triu_indices_from(corr_pred, k=1)
pairs_df = pd.DataFrame({
    "Variable 1": corr_pred.columns[iu[0]],
    "Variable 2": corr_pred.columns[iu[1]],
    "Correlacion (|r|)": corr_pred.values[iu]
})

print(f"Pares con correlación |r| >= 0.90: {(pairs_df['Correlacion (|r|)'] >= 0.90).sum()}")
print(f"Pares con correlación |r| >= 0.95: {(pairs_df['Correlacion (|r|)'] >= 0.95).sum()}")
print(f"Pares con correlación |r| >= 0.99: {(pairs_df['Correlacion (|r|)'] >= 0.99).sum()}")

print("
Top 6 pares con mayor redundancia:")
print(pairs_df.sort_values(by="Correlacion (|r|)", ascending=False).head(6).to_string(index=False))

# Matriz de correlación de las 18 variables más correlacionadas con la quiebra
corr_con_quiebra = df.corr()['Bankrupt?'].abs().sort_values(ascending=False)
top_18_vars = corr_con_quiebra.index[1:19]

plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(df[top_18_vars].corr(), dtype=bool))
sns.heatmap(df[top_18_vars].corr(), mask=mask, cmap='vlag', vmin=-1, vmax=1, annot=True, fmt='.2f', 
            annot_kws={'size': 7}, cbar_kws={'label': 'Coeficiente de Pearson (r)'}, square=True)
plt.title('Figura 3: Matriz de Correlación de las Top 18 Variables más Asociadas a Quiebra', fontsize=11, fontweight='bold', pad=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join("figures", "fig3_matriz_correlaciones.png"), dpi=300, bbox_inches='tight')
plt.show()


### 2.6 Definición y Justificación del Paradigma de Aprendizaje

A partir del análisis exploratorio se establece formalmente el paradigma de modelado:
- **Paradigma:** Aprendizaje Supervisado.
- **Tipo de Problema:** Clasificación Binaria Desbalanceada con Asimetría de Costos ($y_i \in \{0, 1\}$).
- **Justificación:**
  1. Contamos con un conjunto auditado con etiquetas verdaderas proporcionadas por la bolsa de valores (TWSE) a lo largo del periodo 1999-2009.
  2. Los enfoques no supervisados (como agrupamiento o clustering k-means) son inapropiados como modelo de predicción de insolvencia, ya que las empresas en quiebra no conforman un conglomerado denso aislado en el espacio euclidiano, sino instancias periféricas y solapadas en fronteras ruidosas con respecto a las solventes.
  3. El objetivo operativo del sistema es **minimizar el riesgo empírico ponderado por costos**, priorizando la sensibilidad (Recall de quiebras) sin generar una proliferación inmanejable de falsas alarmas.
